In [17]:
import torch 
import numpy as np 
import torch.nn.functional as F 

In [36]:
def kmeans(points: np.ndarray, k: int, eps = 1e-4):
    n = points.shape[0]
    # 1. random init
    # centroids = points[np.random.choice(n, k, replace=False)]

    # kmeans++
    centroids = np.zeros((k, points.shape[1]), dtype=float)
    centroids[0] = points[np.random.choice(n)]
    for i in range(1, k):
        dist = np.linalg.norm(points[:, np.newaxis, :] - centroids[:i], axis=2)
        prob = np.min(dist, axis=1) ** 2
        prob = prob / prob.sum()
        c = np.random.choice(n, p=prob)
        centroids[i] = points[c]

    for _ in range(100):
        # 2. compute labels
        dist = np.linalg.norm(points[:, np.newaxis, :] - centroids, axis=2)
        labels = np.argmin(dist, axis=1)

        # 3. handle empty clusters
        for i in range(k):
            if np.sum(labels == i) == 0:
                c = np.random.choice(n)
                labels[c] = i 
                centroids[i] = points[c] 

        # 4. recompute centroids
        new_centroids = np.array([np.mean(points[labels == i, :], axis=0) for i in range(k)])
        if np.max(np.linalg.norm(new_centroids - centroids, axis=1)) < eps:
            break
        centroids = new_centroids
    
    return labels, centroids

In [37]:
points = np.array([[1,0], [2,0], [3,0], [1,100], [2,100], [4, 100]], dtype=float)
kmeans(points, 2)

(array([1, 1, 1, 0, 0, 0]),
 array([[  2.33333333, 100.        ],
        [  2.        ,   0.        ]]))

In [ ]:
def knn_predict_simple(X_train, y_train, X_test, k=5, task='classification'):
    """
    Simpler vectorized KNN (without scipy dependency).
    Uses manual mode calculation for classification.

    Parameters:
    -----------
    X_train : array-like, shape (n_train_samples, n_features)
        Training data
    y_train : array-like, shape (n_train_samples,)
        Training labels/values
    X_test : array-like, shape (n_test_samples, n_features)
        Test data
    """
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    
    # Compute distances using broadcasting
    # ||X_test[i] - X_train[j]||^2
    distances = np.sqrt(((X_test[:, np.newaxis, :] - X_train[np.newaxis, :, :]) ** 2).sum(axis=2))
    
    # Get k nearest neighbors
    k_nearest_indices = np.argsort(distances, axis=1)[:, :k]
    k_nearest_labels = y_train[k_nearest_indices]
    
    if task == 'classification':
        # Manual mode calculation using bincount
        predictions = np.array([
            np.bincount(k_nearest_labels[i].astype(int)).argmax() 
            for i in range(len(X_test))
        ])
    else:
        predictions = np.mean(k_nearest_labels, axis=1)
    
    return predictions

In [2]:
import torch
import torch.nn as nn 
import torch.nn.functional as F 


In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        assert d_model % n_head == 0
        self.d_model = d_model 
        self.n_head = n_head 
        self.head_size = d_model // n_head 

        self.Q_proj = nn.Linear(d_model, d_model)
        self.K_proj = nn.Linear(d_model, d_model)
        self.V_proj = nn.Linear(d_model, d_model)
        self.O_proj = nn.Linear(d_model, d_model)

    def forward(self, q, k, v):
        bsz, seq_len, _ = q.shape 
        Xq = self.Q_proj(q).view(bsz, seq_len, self.n_head, self.head_size).transpose(1, 2)
        Xk = self.K_proj(k).view(bsz, seq_len, self.n_head, self.head_size).transpose(1, 2)
        Xv = self.V_proj(v).view(bsz, seq_len, self.n_head, self.head_size).transpose(1, 2)

        scores = torch.matmul(Xq, Xk.transpose(-1, -2)) / np.sqrt(self.head_size)
        attn_weights = F.softmax(scores, dim=-1)
        attn_outputs = torch.matmul(attn_weights, Xv)
        attn_outputs = attn_outputs.transpose(1, 2).contiguous.view(bsz, seq_len, -1)
        outputs = self.O_proj(attn_outputs)
        return outputs 



In [5]:
class FFN(nn.Module):
    def __init__(self, d_model, d_fc):
        self.fc1 = nn.Linear(d_model, d_fc)
        self.fc2 = nn.Linear(d_fc, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))
    

In [6]:
class TransformerLayer(nn.Module):
    def __init__(self, d_model, n_head, d_fc):
        self.mha = MultiHeadAttention(d_model, n_head)
        self.ffn = FFN(d_model, d_fc)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        z = self.mha(x, x, x)
        x = self.norm1(x + z)

        z = self.ffn(x)
        x = self.norm2(x + z)
        return x

In [ ]:
import xgboost as xgb
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
X_train = np.random.rand(100, 10)   
y_train = np.random.randint(0, 2, size=100)
model.fit(X_train, y_train)
X_test = np.random.rand(20, 10)
preds = model.predict(X_test)

In [ ]:
import xgboost as xgb
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)
X_train = np.random.rand(100, 10)
y_train = np.random.rand(100)
model.fit(X_train, y_train)
X_test = np.random.rand(20, 10)
preds = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error, accuracy_score
err = mean_squared_error(y_true, y_pred)
acc = accuracy_score(y_true, y_pred)

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
lin_reg = LinearRegression()
log_reg = LogisticRegression()
lin_reg.fit(X_train, y_train)
log_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)
y_pred_log = log_reg.predict(X_test)